# Requirement 5: Missed-Settlement Cluster Analysis

This notebook runs the approved Global and Local Moran's I workflow using PostGIS-resident baseline reconciliation data. It identifies spatial concentration of the GPS-derived missed indicator; it does not establish cause, verify individual visits, or infer vaccination outcomes.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Run this notebook from the Q1 project or notebooks directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\umary\Desktop\EHA test\Q1_Campaign_Team_Tracking


## Approved analysis design

Primary: 2,382 settlements, excluding 180 ambiguous GPS classifications; `missed_indicator=1` for unvisited and `0` for visited. Weights are binary, row-standardized k-nearest neighbours (`k=8`) in EPSG:32632. Sensitivities use a 9,050 m distance band and inclusion of all 2,562 settlements with ambiguous cases treated as missed. Inference uses 999 permutations, seed 20260730, alpha 0.05, and Benjamini-Hochberg FDR adjustment.

In [2]:
from src.spatial_statistics.moran_analysis import execute_requirement_five

results = execute_requirement_five()
results['global']

                               analysis_scenario  ... permutation_p_value
0               primary_knn8_excluding_ambiguous  ...               0.026
1  sensitivity_distance_band_excluding_ambiguous  ...               0.001
2           sensitivity_knn8_ambiguous_as_missed  ...               0.140

[3 rows x 11 columns]

## Local results and interpretation

Raw permutation results are retained for diagnostic transparency. The primary Local Moran labels use FDR-adjusted significance, so multiple local tests are not interpreted as independent discoveries. A non-significant label is not proof of no operational concern.

In [3]:
results['cluster'].sort_values(['analysis_scenario', 'inference', 'cluster_class'])

                                analysis_scenario  ... settlement_count
11               primary_knn8_excluding_ambiguous  ...             2318
0                primary_knn8_excluding_ambiguous  ...                4
1                primary_knn8_excluding_ambiguous  ...               63
2                primary_knn8_excluding_ambiguous  ...             2251
12  sensitivity_distance_band_excluding_ambiguous  ...             2318
3   sensitivity_distance_band_excluding_ambiguous  ...               54
4   sensitivity_distance_band_excluding_ambiguous  ...               40
5   sensitivity_distance_band_excluding_ambiguous  ...               78
6   sensitivity_distance_band_excluding_ambiguous  ...             2146
13           sensitivity_knn8_ambiguous_as_missed  ...             2562
7            sensitivity_knn8_ambiguous_as_missed  ...               10
8            sensitivity_knn8_ambiguous_as_missed  ...               86
9            sensitivity_knn8_ambiguous_as_missed  ...          

## Validation

The execution validates unique settlement IDs within each scenario, valid source geometry, preserved islands, fixed-seed reproducibility, and reconciliation of local class counts to each analysis population. The four CSV tables are written to `outputs/tables/`.

In [4]:
results['diagnostics']

                         weights_specification  ...                              analysis_scenario
0               knn_k4_binary_row_standardized  ...                    primary_excluding_ambiguous
1               knn_k6_binary_row_standardized  ...                    primary_excluding_ambiguous
2               knn_k8_binary_row_standardized  ...                    primary_excluding_ambiguous
3              knn_k10_binary_row_standardized  ...                    primary_excluding_ambiguous
4  distance_band_9050m_binary_row_standardized  ...  sensitivity_distance_band_excluding_ambiguous
5               knn_k8_binary_row_standardized  ...           sensitivity_knn8_ambiguous_as_missed

[6 rows x 10 columns]